In [1]:
# utils.py

import random
import numpy as np
import torch
import os


def set_seed(seed: int = 33):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


SYSTEM_PROMPT = """You are a helpful assistant that writes Python code.
Given a problem description, implement a function that solves it.
Return only the code. Avoid explanations, markdown formatting and test cases."""


def format_prompt(prompt_text: str, system_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Problem: {prompt_text}\n\nImplement the function in Python:",
        },
    ]
    return messages

In [2]:
# prepare_dataset.py

from datasets import load_dataset

def get_train_val_data(train_size: int = 100, val_size: int = 30):
    set_seed()

    dataset = load_dataset("google-research-datasets/mbpp", "sanitized")
    train_dataset = dataset["train"]
    val_dataset = dataset["validation"]

    return train_dataset[:train_size], val_dataset[:val_size]

In [3]:
# verifier.py

from typing import List
import re
import numpy as np


def extract_code_block(text: str) -> str:
    """
    Извлечение кода из markdown, если модель написала с markdown
    """

    pattern = r"```(?:python)?\s*\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    text = re.sub(r"</?answer>|</?think>", "", text)
    return text.strip()


def verify_solution(code: str, test_list: List[str], test_imports: str = "") -> bool:
    """
    Проверяет code, сгенерированный моделью на тестах. Возвращает True если все тесты прошли, False иначе

    code: сгенерированный код функции
    test_list: список assert-выражений (строки)
    test_imports: необходимые импорты для тестов
    """

    code = extract_code_block(code)
    full_code = test_imports + "\n" + code + "\n"

    try:
        namespace = {}
        exec(full_code, namespace)

        for test in test_list:
            if test.strip().startswith("assert "):
                test_expr = test.strip()[7:].strip()  # убираем "assert "
            else:
                test_expr = test.strip()

            result = eval(test_expr, namespace)
            if not result:
                return False
        return True

    except Exception as e:
        return False


def pass_k(n: int, c: int, k: int) -> float:
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def calc_pass_k(num_samples: List[int], num_correct: List[int], k: int) -> float:
    """
    Подсчет метрики по всему набору задач
    """

    pass_values = [pass_k(n, c, k) for n, c in zip(num_samples, num_correct)]
    return np.mean(pass_values)

In [4]:
# baseline_evaluation.py

from typing import Tuple
import torch
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")

# from utils import set_seed, format_prompt, SYSTEM_PROMPT
# from prepare_dataset import get_train_val_data
# from verifier import verify_solution


def evaluate_task(model, tokenizer, task: dict, config: dict) -> int:
    """
    Генерирует config['num_samples'] решений и возвращает количество корректных решений
    """

    # Передадим явно название функции из тесткейсов
    required_func_name = task["test_list"][0][7:].split("(")[0]
    prompt_text = (
        task["prompt"] + f"Give the function a name like this: {required_func_name}."
    )
    test_list = task["test_list"]
    test_imports = "\n".join(task["test_imports"])
    messages = format_prompt(prompt_text, SYSTEM_PROMPT)

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(config["device"])
    
    input_ids = inputs["input_ids"].repeat(config["num_samples"], 1)
    attention_mask = inputs["attention_mask"].repeat(config["num_samples"], 1)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            temperature=config["temperature"],
            top_k=config["top_k"],
            max_new_tokens=config["max_new_tokens"],
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    correct_count = 0
    prompt_length = inputs["input_ids"].shape[1]
    
    for i in range(config["num_samples"]):
        generated = tokenizer.decode(
            outputs[i][prompt_length:], skip_special_tokens=True
        )
        # print(generated)
        if verify_solution(generated, test_list, test_imports):
            correct_count += 1
    
    return correct_count


def main():
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    CONFIG = {
        "num_samples": 10,
        "device": device,
        "temperature": 1.0,
        "top_k": 50,
        "max_new_tokens": 256,
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
    }
    print(f"Запуск с параметрами: {CONFIG}")

    model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], torch_dtype=torch.float16, device_map=device)
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])

    train_data, val_data = get_train_val_data()

    train_hard_idx, val_hard_idx = [], []
    data_for_curves_train, data_for_curves_val = [], []
    train_length, val_length = len(train_data["prompt"]), len(val_data["prompt"])

    print("Начало baseline train оценки...")
    for i in tqdm(range(train_length)):
        task = {
            "prompt": train_data["prompt"][i],
            "test_list": train_data["test_list"][i],
            "test_imports": train_data["test_imports"][i],
        }
        correct_count = evaluate_task(model, tokenizer, task, CONFIG)
        data_for_curves_train.append(correct_count)

        if correct_count == 0:
            print(
                f"{i}/{train_length} пример подходит, ни одного правильного ответа из {CONFIG['num_samples']}"
            )
            train_hard_idx.append(i)

    print("\n\nНачало baseline val оценки...")
    for i in tqdm(range(val_length)):
        task = {
            "prompt": val_data["prompt"][i],
            "test_list": val_data["test_list"][i],
            "test_imports": val_data["test_imports"][i],
        }
        correct_count = evaluate_task(model, tokenizer, task, CONFIG)
        data_for_curves_val.append(correct_count)

        if correct_count == 0:
            print(
                f"{i}/{val_length} пример подходит, ни одного правильного ответа из {CONFIG['num_samples']}"
            )
            val_hard_idx.append(i)

    print(
        f"\n\nBaseline оценка завершена. Найдено train_hard_idx: {len(train_hard_idx)}, val_hard_idx: {len(val_hard_idx)}"
    )

    data = {"train_hard_idx": train_hard_idx, "val_hard_idx": val_hard_idx}
    with open("hard_idx.json", "w", encoding="utf-8") as json_file:
        json.dump(data, json_file, ensure_ascii=False, indent=4)

    data = {
        "data_for_curves_train": data_for_curves_train,
        "data_for_curves_val": data_for_curves_val,
    }
    with open("data_for_curves.json", "w", encoding="utf-8") as json_file:
        json.dump(data, json_file, ensure_ascii=False, indent=4)


if __name__ == "__main__":
    main()

Запуск с параметрами: {'num_samples': 10, 'device': 'cuda', 'temperature': 1.0, 'top_k': 50, 'max_new_tokens': 256, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Начало baseline train оценки...


  1%|          | 1/100 [00:13<21:37, 13.11s/it]

...

True
False
False
True
False
True
False
False
True
False


  5%|▌         | 5/100 [00:58<18:20, 11.58s/it]

90 degrees is equal to 1.57 radians.
1.5707963267948966


  5%|▌         | 5/100 [01:05<20:49, 13.15s/it]


KeyboardInterrupt: 